# Data Science Lifecycle: Part 3 - Feature Engineering

Transforming raw continuous and categorical features into high-signal variables to simplify model optimization and enhance R² performance.

### Objectives:
1. **Time Features**: Extract temporal signals (`is_peak_hour`, `is_weekend`, `is_night`).
2. **Distance Buckets**: Construct ordinal distance categorical buckets (`Short`, `Medium`, `Long`, `Very Long`).
3. **Traffic Severity Score**: Translate subjective traffic levels into continuous impact factors ($1 \rightarrow 4$).
4. **Weather Severity Score**: Translate subjective weather states into continuous impact factors ($1 \rightarrow 4$).

---

In [ ]:
import pandas as pd
import numpy as np

processed_csv_path = "../data/processed/processed_deliveries.csv"
df = pd.read_csv(processed_csv_path)
df.head()

## 1. Traffic and Weather Severity Scoring

Subjective levels can be represented numerically to give linear algorithms direct quantitative signal of the delay magnitude.

In [ ]:
traffic_map = {"Low": 1, "Medium": 2, "High": 3, "Jam": 4}
weather_map = {"Sunny": 1, "Cloudy": 2, "Rainy": 3, "Rain": 3, "Storm": 4}

df["Traffic_Severity_Score"] = df["Traffic_Level"].map(traffic_map)
df["Weather_Severity_Score"] = df["Weather"].map(weather_map)

df[["Traffic_Level", "Traffic_Severity_Score", "Weather", "Weather_Severity_Score"]].head()

## 2. Distance Buckets Creation

Segmenting distances into ranges allows tree models to quickly isolate short-haul local dispatch vs extreme delivery routes.

In [ ]:
def map_distance_bucket(dist):
    if dist <= 3.0: return "Short"
    elif dist <= 8.0: return "Medium"
    elif dist <= 15.0: return "Long"
    else: return "Very Long"

df["Distance_Bucket"] = df["Distance_km"].apply(map_distance_bucket)
print("Distance Bucket Counts:")
print(df["Distance_Bucket"].value_counts())

## 3. Temporal Extraction flags

In [ ]:
# Weekend flag
df["is_weekend"] = df["Day_of_Week"].apply(lambda x: 1 if str(x).capitalize() in ["Saturday", "Sunday"] else 0)

# Night flag
df["is_night"] = df["Time_of_Day"].apply(lambda x: 1 if str(x).capitalize() == "Night" else 0)

# Peak hour flag
df["is_peak_hour"] = df["Peak_Hour"].apply(lambda x: 1 if str(x).lower() == "yes" else 0)

df[["Day_of_Week", "is_weekend", "Time_of_Day", "is_night", "Peak_Hour", "is_peak_hour"]].head()

## 4. Feature Signal Verification

In [ ]:
corr_with_target = df[[
    "Distance_km", "Preparation_Time", "Traffic_Severity_Score", 
    "Weather_Severity_Score", "is_peak_hour", "is_weekend", "is_night"
]].corrwith(df["Delivery_Time_Min"])

print("Engineered continuous features correlation with target (Delivery_Time_Min):")
print(corr_with_target.sort_values(ascending=False))